# CS F407 — SegNet Training on Kaggle
Run via **Save Version → Save & Run All (Commit)** for background execution.

In [ ]:
import os, sys, glob, shutil, json

WORKING = '/kaggle/working'

# ── 1. Find src .py files and copy to working dir ────────────────────────────
py_files = glob.glob('/kaggle/input/**/*.py', recursive=True)
if not py_files:
    raise RuntimeError('No .py files found in /kaggle/input — add your src dataset')

for f in py_files:
    shutil.copy2(f, os.path.join(WORKING, os.path.basename(f)))
    print(f'  copied {os.path.basename(f)}')

# ── 2. Create mission_config.json directly (avoids dataset version issues) ───
mission = {
    'mission_id': 'OP-DELTA-9',
    'package': {
        'type': 'medical_vials',
        'heavy': True,
        'fragile': False,
        'valuable': False,
        'biohazard': False
    }
}
with open(os.path.join(WORKING, 'mission_config.json'), 'w') as f:
    json.dump(mission, f, indent=4)
print('  created mission_config.json')

print('\nSrc setup done.')

In [ ]:
# ── 3. Find training data and link it into working dir ───────────────────────
# Searches all input datasets for the images/ folder.
images_candidates = glob.glob('/kaggle/input/**/images', recursive=True)
images_candidates = [p for p in images_candidates
                     if len(glob.glob(os.path.join(p, '*.jpg'))) > 10]

if not images_candidates:
    raise RuntimeError('Could not find images/ folder — add your training_set dataset')

images_dir      = images_candidates[0]
training_set_src = os.path.dirname(images_dir)   # parent of images/
training_set_dst = os.path.join(WORKING, 'training_set')

if not os.path.exists(training_set_dst):
    os.symlink(training_set_src, training_set_dst)
    print(f'Symlinked training_set: {training_set_src}')
else:
    print(f'training_set already linked')

n_img   = len(glob.glob(os.path.join(training_set_dst, 'images', '*.jpg')))
n_masks = len(glob.glob(os.path.join(training_set_dst, 'gt', 'semantic', 'label_images', '*.png')))
print(f'Images : {n_img}')
print(f'Masks  : {n_masks}')
if n_img == 0 or n_masks == 0:
    raise RuntimeError('Images or masks not found — check dataset structure')

In [ ]:
# ── 4. Verify GPU ─────────────────────────────────────────────────────────────
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    raise RuntimeError('No GPU detected — set Accelerator to GPU T4 in Session options')

In [ ]:
# ── 5. Run training ───────────────────────────────────────────────────────────
# Use shell=True so Python adds /kaggle/working to sys.path automatically
# (same behaviour as running `python train.py` from the terminal).
import subprocess

result = subprocess.run(
    f'{sys.executable} train.py',
    cwd=WORKING,
    shell=True,
    check=False
)

print(f'\nProcess exited with code: {result.returncode}')
if result.returncode != 0:
    raise RuntimeError('Training failed — scroll up for error details')

In [ ]:
# ── 6. Verify output ─────────────────────────────────────────────────────────
ckpt = os.path.join(WORKING, 'best_model.pth')
if not os.path.isfile(ckpt):
    raise RuntimeError('best_model.pth not found — training may have failed')

data  = torch.load(ckpt, map_location='cpu', weights_only=False)
mb    = os.path.getsize(ckpt) / 1e6
print(f'best_model.pth  ({mb:.1f} MB)')
print(f'  Best epoch  : {data["epoch"]}')
print(f'  Val loss    : {data["best_val_loss"]:.4f}')
print(f'  Val acc     : {data["val_metrics"]["pixel_acc"]*100:.2f}%')
print(f'  Val mIoU    : {data["val_metrics"]["mean_iou"]*100:.2f}%')
print('\nDownload best_model.pth from the Output panel on the right.')